# KG1 v73 — GRPO TRL Stage-2 (Colab Pro+ A100)

## Framework: priyanlc/autoresearch-sft-grpo + dtdo90 rewards

**Bombas**:
- 6 rewards compostos (priyanlc): correctness + format + reasoning + category_bonus + single_box + final_line
- KL beta=0.01, num_generations=4, LR=5e-6
- Foco: bit_manip 3-input + cryptarithm + equation_guess (gaps huikang)
- Memory: ~36-38GB A100 (vllm rollouts otimizados)

## Pré-requisito: V73 SFT adapter (FASE 2 completa) em Drive ou HF

## Score esperado: 0.86 → 0.87 (P=70%+)

In [ ]:
# Cell 1: Setup + clone repo
import torch, subprocess, os, sys
r = subprocess.run('nvidia-smi', shell=True, capture_output=True, text=True)
print(r.stdout[:800])

from IPython.display import display, Javascript
display(Javascript("function ClickConnect(){document.querySelector('colab-connect-button').click()};setInterval(ClickConnect, 60000)"))

%pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'
%pip install -q --no-deps 'trl>=0.16' 'peft>=0.18.1' accelerate bitsandbytes
%pip install -q vllm 'transformers>=4.55' datasets liger-kernel

# Verify TRL version
import trl
assert trl.__version__ >= '0.16', f'TRL {trl.__version__} < 0.16 (GRPO requires processing_class kwarg)'
print(f'TRL {trl.__version__} OK')

# Clone repo (para src/competition_utils.py) usando os.system para parseability
if not os.path.exists('/content/kg1'):
    rc = os.system('git clone https://github.com/FELIPEACASTRO/KG1.git /content/kg1 2>/dev/null')
    if rc != 0:
        print('WARN: repo clone falhou, usando inline fallback')
sys.path.insert(0, '/content/kg1/src')
sys.path.insert(0, '/content/kg1')

from google.colab import drive, userdata
drive.mount('/content/drive')

try:
    HF_TOKEN = userdata.get('HF_KEY')
except Exception:
    HF_TOKEN = userdata.get('HF_TOKEN', '')
assert HF_TOKEN.startswith('hf_'), 'Configure HF_KEY no Colab Secrets'
os.environ['HF_TOKEN'] = HF_TOKEN


In [ ]:
# Cell 2: Load V73 SFT adapter (FASE 2) como ponto de partida
from unsloth import FastLanguageModel

MAX_SEQ = 8192  # Kaggle metric usa max_model_len=8192. GRPO precisa max_prompt+max_completion <= MAX_SEQ
model, tok = FastLanguageModel.from_pretrained(
    model_name='unsloth/Nemotron-3-Nano-30B-A3B-bnb-4bit',
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    full_finetuning=False,
    token=HF_TOKEN,
)

# Carregar V73 SFT adapter como base
from peft import PeftModel
V73_SFT_PATH = '/content/drive/MyDrive/kg1_v73_unsloth_moe/final_adapter'
if not os.path.exists(V73_SFT_PATH):
    # Fallback: download from HF
    from huggingface_hub import snapshot_download
    V73_SFT_PATH = snapshot_download(
        'felipesp1983/kg1-nemotron-lora-v73-unsloth-moe',
        token=HF_TOKEN, allow_patterns=['final/*']
    ) + '/final'

model = PeftModel.from_pretrained(model, V73_SFT_PATH, adapter_name='sft', is_trainable=True)
model.set_adapter('sft')
print(f'Loaded V73 SFT from {V73_SFT_PATH}')
print(f'Active adapter: {model.active_adapters() if hasattr(model, "active_adapters") else "sft"}')


In [ ]:
# Cell 3: Reward functions (TRL 0.16+ kwargs vem das colunas do dataset)
import re

# Inline competition_utils (fallback se import do repo falhou)
try:
    from competition_utils import extract_final_answer, verify
    print('Using competition_utils from repo')
except ImportError:
    print('Using inline extract_final_answer/verify (fallback)')
    BOXED_INNER = re.compile(r'\\boxed\{([^{}]+)\}')
    def extract_final_answer(text):
        if not isinstance(text, str): return None
        m = BOXED_INNER.findall(text)
        return m[-1].strip() if m else None
    def verify(answer, predicted):
        if not predicted: return False
        a, p = str(answer).strip(), str(predicted).strip()
        if re.fullmatch(r'[01]+', a):
            return a == p
        return a == p

BOXED_RE = re.compile(r'\\boxed\{([^{}]+)\}')
FINAL_LINE_RE = re.compile(r'(?:Answer|Final|Result|Resposta)[:\s]+([^\n]+)', re.I)

W_CORRECTNESS = 1.0
W_FORMAT = 0.3
W_REASONING = 0.15
W_CATEGORY_BONUS = 0.2
W_SINGLE_BOX = 0.08
W_FINAL_LINE = 0.02

# CRITICO: TRL passa kwargs com NOME EXATO da coluna do dataset
# Dataset (pos to_grpo) tem colunas: prompt, answer (singular), family (singular)

def _to_text(c):
    return c if isinstance(c, str) else (c[0]['content'] if isinstance(c, list) else str(c))

def _normalize_family(fam):
    '''Normaliza family para matching consistente (case + spaces).'''
    return str(fam).lower().replace(' ', '_').replace('-', '_')

def reward_correctness(prompts, completions, answer, **kwargs):
    rewards = []
    for c, a in zip(completions, answer):
        pred = extract_final_answer(_to_text(c))
        rewards.append(W_CORRECTNESS if verify(str(a), str(pred)) else 0.0)
    return rewards

def reward_format(prompts, completions, **kwargs):
    return [W_FORMAT if BOXED_RE.search(_to_text(c)) else 0.0 for c in completions]

def reward_single_box(prompts, completions, **kwargs):
    return [W_SINGLE_BOX if len(BOXED_RE.findall(_to_text(c))) == 1 else 0.0 for c in completions]

def reward_final_line(prompts, completions, **kwargs):
    return [W_FINAL_LINE if FINAL_LINE_RE.search(_to_text(c)) else 0.0 for c in completions]

def reward_reasoning(prompts, completions, **kwargs):
    rewards = []
    for c in completions:
        L = len(_to_text(c))
        if 500 <= L <= 3000:
            rewards.append(W_REASONING)
        elif L > 3000:
            rewards.append(W_REASONING * 0.5)
        else:
            rewards.append(0.0)
    return rewards

# Markers por keyword de familia (robusto a naming variants)
FAMILY_MARKERS = {
    'bit':         ['XOR', 'AND', 'OR', 'NOT', 'binary', '01'],
    'cipher':      ['substitution', 'mapping', 'decrypt', 'letter'],
    'encryption':  ['substitution', 'mapping', 'decrypt', 'letter'],
    'crypt':       ['letter', 'digit', 'maps'],
    'equation':    ['operator', 'arithmetic', '+', '-', '*', '/'],
    'gravity':     ['g =', 'd =', 't =', 'gravitational'],
    'unit':        ['multiply', 'convert', 'km', 'mph'],
    'numeral':     ['roman', 'decimal'],
}

def reward_category_bonus(prompts, completions, family, **kwargs):
    rewards = []
    for c, fam in zip(completions, family):
        fam_norm = _normalize_family(fam)
        # Pega primeiro keyword que da match
        ms = []
        for kw, markers in FAMILY_MARKERS.items():
            if kw in fam_norm:
                ms = markers
                break
        if not ms:
            rewards.append(0.0)
            continue
        text = _to_text(c).lower()
        hits = sum(1 for m in ms if m.lower() in text)
        rewards.append(W_CATEGORY_BONUS * min(1.0, hits / len(ms)))
    return rewards

REWARD_FUNCS = [reward_correctness, reward_format, reward_single_box,
                reward_final_line, reward_reasoning, reward_category_bonus]
print(f'Loaded {len(REWARD_FUNCS)} reward functions')


In [ ]:
# Cell 4: Dataset GRPO - train.csv OFICIAL (subset HARD)
# Alinhado com V73 UNSLOTH (kienngx recipe) - usa train.csv oficial, nao huikang-16k
import pandas as pd
from datasets import Dataset

# Baixar train.csv oficial Kaggle (via Drive ou API)
TRAIN_CSV = '/content/drive/MyDrive/kg1_train.csv'
if not os.path.exists(TRAIN_CSV):
    os.makedirs('/root/.kaggle', exist_ok=True)
    import shutil, json as _json
    kaggle_json_drive = '/content/drive/MyDrive/.kaggle/kaggle.json'
    if os.path.exists(kaggle_json_drive):
        shutil.copy(kaggle_json_drive, '/root/.kaggle/kaggle.json')
    else:
        try:
            from google.colab import userdata
            kaggle_user = userdata.get('KAGGLE_USERNAME')
            kaggle_key = userdata.get('KAGGLE_KEY')
            with open('/root/.kaggle/kaggle.json', 'w') as f:
                _json.dump({'username': kaggle_user, 'key': kaggle_key}, f)
        except Exception as e:
            raise RuntimeError(f'Configure KAGGLE creds: {e}')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    os.system('kaggle competitions download -c nvidia-nemotron-model-reasoning-challenge -f train.csv -p /content/')
    os.system('unzip -o /content/train.csv.zip -d /content/ 2>/dev/null || true')
    TRAIN_CSV = '/content/train.csv'

df_full = pd.read_csv(TRAIN_CSV)
print(f'Train.csv: {len(df_full)} rows | columns: {list(df_full.columns)}')

# Distribuicao por categoria (se existir)
CAT_COL = 'category' if 'category' in df_full.columns else ('family' if 'family' in df_full.columns else None)
if CAT_COL:
    print('Distribuicao por categoria:')
    print(df_full[CAT_COL].value_counts())

# HARD subset: categorias onde nosso pipeline historico teve menor score
# kienngx base chegou 0.86 sem filtro; GRPO vai aprender especificamente nos HARDs
HARD_KEYWORDS = ['bit', 'crypt', 'equation', 'cipher', 'encryption']
def _is_hard(cat):
    if not isinstance(cat, str):
        return False
    cat_l = cat.lower().replace(' ', '_')
    return any(kw in cat_l for kw in HARD_KEYWORDS)

if CAT_COL:
    df_hard = df_full[df_full[CAT_COL].apply(_is_hard)].copy()
else:
    # Sem categoria, usa full
    df_hard = df_full.copy()
print(f'Hard subset: {len(df_hard)} (bit + crypt + equation + cipher)')

# Limitar para 600 prompts GRPO (priyanlc scale)
N_GRPO = min(600, len(df_hard))
df_grpo = df_hard.sample(n=N_GRPO, random_state=42).reset_index(drop=True)
print(f'GRPO sample: {N_GRPO}')

# Detect columns
PROMPT_COL = 'prompt' if 'prompt' in df_grpo.columns else 'problem'
ANSWER_COL = 'answer' if 'answer' in df_grpo.columns else 'solution'

def to_grpo(row):
    return {
        'prompt': str(row[PROMPT_COL]),
        'answer': str(row[ANSWER_COL]) if ANSWER_COL in row else '',
        'family': str(row.get(CAT_COL, 'unknown')) if CAT_COL else 'unknown',
    }

ds_grpo = Dataset.from_pandas(df_grpo).map(to_grpo, remove_columns=list(df_grpo.columns))
print(f'GRPO ready: {len(ds_grpo)} prompts | columns: {ds_grpo.column_names}')
print(f'Sample prompt[:100]: {ds_grpo[0]["prompt"][:100]!r}')
print(f'Sample answer: {ds_grpo[0]["answer"][:50]!r}')
print(f'Sample family: {ds_grpo[0]["family"]}')


In [ ]:
# Cell 5: GRPO Training (TRL 0.16+ API)
from trl import GRPOTrainer, GRPOConfig

CKPT_DIR = '/content/drive/MyDrive/kg1_v73_grpo'
os.makedirs(CKPT_DIR, exist_ok=True)

# CRITICO: max_prompt + max_completion deve ser <= MAX_SEQ do model
# 2048 + 4096 = 6144 < 8192 OK
grpo_args = GRPOConfig(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_generations=4,
    max_prompt_length=2048,
    max_completion_length=4096,
    learning_rate=5e-6,
    beta=0.01,
    num_train_epochs=1,
    save_steps=50,
    logging_steps=5,
    bf16=True,
    report_to='none',
    push_to_hub=False,
    seed=42,
    use_vllm=False,
    remove_unused_columns=False,
)

# Diagnostico pre-treino
print(f'Model max_seq: {MAX_SEQ}')
print(f'GRPO max_prompt + max_completion: {grpo_args.max_prompt_length + grpo_args.max_completion_length}')
assert grpo_args.max_prompt_length + grpo_args.max_completion_length <= MAX_SEQ, \
    f'OVERFLOW: prompt+completion > MAX_SEQ ({MAX_SEQ})'
print('OK: prompt+completion fits in MAX_SEQ')

trainer = GRPOTrainer(
    model=model,
    reward_funcs=REWARD_FUNCS,
    args=grpo_args,
    train_dataset=ds_grpo,
    processing_class=tok,
)

resume = None
if os.path.exists(CKPT_DIR):
    ckpts = [d for d in os.listdir(CKPT_DIR) if d.startswith('checkpoint-')]
    if ckpts:
        resume = True
        print(f'Resuming from existing checkpoint')

trainer.train(resume_from_checkpoint=resume)
trainer.save_model(f'{CKPT_DIR}/final_grpo')
print('GRPO done')


In [ ]:
# Cell 6: Upload V73-GRPO adapter
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
REPO_ID = 'felipesp1983/kg1-nemotron-lora-v73-grpo'
api.create_repo(REPO_ID, private=True, exist_ok=True)
api.upload_folder(folder_path=f'{CKPT_DIR}/final_grpo', repo_id=REPO_ID, path_in_repo='final')
print(f'Uploaded V73-GRPO to {REPO_ID}')